# M1b — KG-Localized Surgical Sub-Concept SAE Edit with Side-Effect Measurement

**The downstream payoff of treating SAE features as a knowledge representation.**
Given a feature **knowledge-graph (KG)** that names *one absorber latent per sub-context*, we
edit **exactly one sub-context** by ablating its single named latent
(`h ← h − λ·z_l·W_dec[l]`, gated by that latent's own sparse firing) and show a **high
on-target behavioral effect with near-zero collateral** on sibling sub-contexts — a surgical
capability the standard non-SAE handle (a dense diff-of-means *parent* direction) structurally
cannot provide.

Each edit operator is compared **at matched on-target effect**:

| Operator | What it does | Expected |
|---|---|---|
| **KG-ABL** (ours) | ablate the KG-named absorber latent (token-gated) | high on-target, ~0 collateral, tiny footprint |
| **DENSE-ABL** (non-SAE baseline) | erase the diff-of-means *parent* hyperplane (one direction for the WHOLE parent) | high collateral, footprint ≈ 1 |
| **RAND** | ablate a random firing-rate-matched latent | on-target ≈ 0 |
| **(k)** | label-free probe → dense hyperplane | no per-sub-context latent to edit (structural) |

**Primary measure:** per-context **next-token KL divergence at the edited token's position**.
`SURGICAL SELECTIVITY = on_target / collateral` at matched effect, with **paired bootstrap CIs**.
A graded verdict separates a *clean* surgical edit from a *partial / co-firing* edit, and a
**firing-Jaccard router** predicts which regime each case is in.

---

### What this demo does

The full `method.py` runs `gemma-2-2b` + a Gemma-Scope JumpReLU SAE on a GPU to produce the
behavioral KL curves. This notebook is **$0 / CPU-only**: it loads the **precomputed per-case
curves + per-context KL values** and faithfully **re-runs the analysis layer** with the original
code — re-deriving the matched-effect selectivity, the graded verdict, the bootstrap CIs, and the
regime router map — then visualizes the headline result. The reused functions
(`bootstrap_mean_ci`, `paired_bootstrap_diff`, `_interp_at`, `_scale_for_on_target`, the matched +
verdict logic) are copied verbatim from `method.py`.

In [ ]:
# --- Dependencies (Colab-safe install pattern) ---
# This analysis demo needs only numpy + matplotlib. Both are pre-installed on Colab, so we
# install them ONLY when running locally (and pin to Colab's versions to mirror that env).
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'matplotlib==3.10.0')

In [ ]:
# --- Imports (subset of method.py imports relevant to the CPU analysis, + matplotlib) ---
import json, os
from collections import defaultdict
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# --- Data loading helper: GitHub URL with local fallback (Colab-compatible) ---
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-7ee30c-catching-silent-feature-absorption-in-fr/main/round-4/experiment-2/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
print("Loaded:", data["method_name"])
print("SAE  :", data["sae"]["release"], "|", data["sae"]["sae_params"])
print("Model:", data["model"], "| hook:", data["sae"]["hook"])
print("per-case units :", len(data["per_case"]))
print("Georgia per-context rows:", len(data["edit_locality_per_context_georgia"]))

## Configuration

All tunable parameters live here. They start at small/demo values; the comments give the original
`method.py` values for a faithful full run. Because the analysis operates on tiny precomputed
arrays (≤ ~180 contexts per case), even the original `B_BOOT = 10000` finishes in well under a
second — so scaling up here is essentially free.

In [ ]:
# ---------------------------------------------------------------- DEMO CONFIG
SEED = 1234                        # method.py SEED

# Bootstrap iterations for the paired/mean CIs. The arrays are tiny (<=180 contexts), so the
# original full value runs in well under a second -> we use it directly. (Drop to e.g. 500 for
# an even snappier demo; results are stable either way.)
B_BOOT = 10000                     # == data["B_boot"] (original method.py value)

EPS = 1e-8                         # method.py EPS (selectivity denominator floor)

MAX_CASES = 7                      # number of per-case units to analyse (all 7 are tiny)

# matched-effect + graded-verdict thresholds (verbatim from method.py run_case)
MATCH_FRACTION = 0.8               # matched on-target = MATCH_FRACTION * min(max on-target of both ops)
SEL_CLEAN_THRESHOLD = 20           # KG selectivity >= 20 for a CLEAN surgical edit
FOOTPRINT_CLEAN_THRESHOLD = 0.05   # off-target token footprint < 5%
RATIO_CLEAN_THRESHOLD = 20         # KG/dense selectivity ratio >= 20

# RNG used by the bootstrap routines (method.py: rng = np.random.default_rng(SEED))
rng = np.random.default_rng(SEED)
print(f"B_BOOT={B_BOOT}  SEED={SEED}  MAX_CASES={MAX_CASES}")

## Statistics: paired & mean bootstrap CIs

Copied **verbatim** from `method.py`. `bootstrap_mean_ci` gives a percentile CI for a mean (used
for the on-target effect and each operator's collateral); `paired_bootstrap_diff` gives a CI for
the *dense − kg collateral difference* — the claim "KG is strictly more surgical than dense" holds
iff that CI excludes 0 favoring KG.

In [ ]:
# Paired + mean bootstrap CIs (verbatim from method.py)
def paired_bootstrap_diff(a, b, B=B_BOOT):
    a = np.asarray(a, float); b = np.asarray(b, float)
    n = len(a)
    if n == 0:
        return {"diff": 0.0, "ci_lo": 0.0, "ci_hi": 0.0, "excl_0": False, "n": 0}
    idx = rng.integers(0, n, size=(B, n))
    d = a[idx].mean(1) - b[idx].mean(1)
    lo, hi = np.percentile(d, [2.5, 97.5])
    return {"diff": float(a.mean() - b.mean()), "ci_lo": float(lo), "ci_hi": float(hi),
            "excl_0": bool(lo > 0 or hi < 0), "n": int(n)}


def bootstrap_mean_ci(values, B=B_BOOT):
    v = np.asarray(values, float); n = len(v)
    if n == 0:
        return {"mean": 0.0, "ci_lo": 0.0, "ci_hi": 0.0, "n": 0}
    idx = rng.integers(0, n, size=(B, n))
    bs = v[idx].mean(1)
    lo, hi = np.percentile(bs, [2.5, 97.5])
    return {"mean": float(v.mean()), "ci_lo": float(lo), "ci_hi": float(hi), "n": int(n),
            "excl_0": bool(lo > 0 or hi < 0)}

## Matched-effect interpolation helpers

Copied **verbatim** from `method.py`. To compare operators fairly we read each operator's
collateral at a *common* on-target level, interpolating along its measured curve.

In [ ]:
def _interp_at(xs, ys, x0):
    xs = np.asarray(xs, float); ys = np.asarray(ys, float)
    order = np.argsort(xs)
    return float(np.interp(x0, xs[order], ys[order]))


def _scale_for_on_target(scales, on_curve, target):
    """smallest scale whose interpolated on_target reaches `target` (monotone-ish)."""
    s = np.asarray(scales, float); o = np.asarray(on_curve, float)
    order = np.argsort(o)
    return float(np.interp(target, o[order], s[order]))

## Step 1 — Re-derive the matched-effect **selectivity** from each case's curves

For every case we have the stored behavioral KL **curves** (mean on-target and mean collateral
vs. edit scale) for KG-ABL and DENSE-ABL. Following `run_case` in `method.py`:

1. pick the matched on-target level `target_on = 0.8 · min(max KG on-target, max DENSE on-target)`,
2. read each operator's collateral at that level (interpolated),
3. `selectivity = on_target / |collateral|`, and `ratio = sel(KG) / sel(DENSE)`.

The re-derived ratio is cross-checked against the value `method.py` stored.

In [ ]:
per_case = data["per_case"][:MAX_CASES]
matched_demo = []
for c in per_case:
    cur = c["curves"]
    kg_on_c = np.array(cur["KG-ABL"]["beh_on_target"])
    de_on_c = np.array(cur["DENSE-ABL"]["beh_on_target"])
    # matched on-target effect both operators can attain (run_case logic)
    target_on = max(1e-4, min(float(kg_on_c.max()), float(de_on_c.max())) * MATCH_FRACTION)
    row = {"case": f"{c['family']}/{c['target_subcontext']}", "absorber": c["absorber_latent"],
           "regime": c["regime"], "target_on": target_on}
    for m in ("KG-ABL", "DENSE-ABL"):
        on_c = cur[m]["beh_on_target"]; col_c = cur[m]["beh_collateral"]
        col = _interp_at(on_c, col_c, target_on)
        row[f"sel_{m}"] = target_on / (abs(col) + EPS)
        row[f"col_{m}"] = col
    row["ratio"] = row["sel_KG-ABL"] / max(row["sel_DENSE-ABL"], EPS)
    row["stored_ratio"] = c["headline_selectivity_ratio"]
    matched_demo.append(row)

print(f"{'case':<28}{'absorber':>9}{'regime':>12}{'sel_KG':>12}{'sel_DENSE':>11}{'ratio':>12}{'stored':>12}")
for r in matched_demo:
    print(f"{r['case']:<28}{r['absorber']:>9}{r['regime']:>12}"
          f"{r['sel_KG-ABL']:>12.1f}{r['sel_DENSE-ABL']:>11.3f}{r['ratio']:>12.1f}{r['stored_ratio']:>12.1f}")

## Step 2 — Re-derive the graded **verdict**

Using the exact decision rule from `method.py run_case`. A **CLEAN** surgical edit requires:
KG selectivity ≥ 20, off-target token footprint < 5%, and KG/dense ratio ≥ 20 — together with a
real on-target effect (`on_ci` excludes 0) and KG strictly beating dense on collateral
(`dense − kg collateral` CI excludes 0). The co-firing regime is *expected* to be non-clean.

In [ ]:
verdict_demo = []
for c, r in zip(per_case, matched_demo):
    ci = c["selectivity_CIs"]; mt = c["matched"]; regime = c["regime"]
    sel_kg = mt["KG-ABL"]["selectivity"]
    ratio = c["headline_selectivity_ratio"]
    fp_off = mt["KG-ABL"].get("token_footprint_offtarget", 1.0)
    on_ci = ci["KG-ABL_on_target"]
    col_diff_ci = ci["dense_minus_kg_collateral"]
    on_ok = bool(on_ci.get("excl_0") and on_ci["mean"] > 0)          # real on-target effect
    dominates = bool(col_diff_ci["excl_0"] and col_diff_ci["diff"] > 0)  # KG < dense collateral
    clean = bool(sel_kg >= SEL_CLEAN_THRESHOLD and fp_off < FOOTPRINT_CLEAN_THRESHOLD
                 and ratio >= RATIO_CLEAN_THRESHOLD)
    if on_ok and dominates and clean:
        verdict = "SURGICAL_EDIT_CONFIRMED"
    elif on_ok and dominates:
        verdict = "PARTIAL_CO_FIRING_AS_PREDICTED" if regime == "co-firing" else "PARTIAL_SURGICAL"
    elif on_ok and not dominates:
        verdict = "NO_SELECTIVITY_ADVANTAGE"
    else:
        verdict = "NO_ON_TARGET_EFFECT"
    ok = (verdict == c["verdict"])
    verdict_demo.append({"case": r["case"], "verdict": verdict})
    print(f"{r['case']:<28} -> {verdict:<32} (stored {c['verdict']:<32}) {'OK' if ok else 'MISMATCH'}")

n_surg = sum(v["verdict"] == "SURGICAL_EDIT_CONFIRMED" for v in verdict_demo)
print(f"\n{n_surg}/{len(verdict_demo)} cases SURGICAL_EDIT_CONFIRMED")

## Step 3 — Replay the bootstrap CIs on real per-context KL (Georgia)

For the headline **taxonomic / Georgia** case (absorber **16009**) we ship the per-context KL
values at the *full* edit (λ=1 / β=1). Each row is one held-out context labelled `ON_TARGET`
(a Georgia context the edit *should* change) or `OFF_TARGET_SIBLING` (a sibling it should *not*).
We feed the real KL arrays through the verbatim bootstrap routines:

- KG-ABL on-target KL should be **large** and its CI exclude 0,
- KG-ABL collateral (siblings) should be **≈ 0**,
- DENSE collateral should be **large**, and the `dense − kg` collateral difference CI exclude 0.

In [ ]:
geo = data["edit_locality_per_context_georgia"]
on_rows  = [r for r in geo if r["output"] == "ON_TARGET"]
sib_rows = [r for r in geo if r["output"] == "OFF_TARGET_SIBLING"]
kg_on  = np.array([r["metadata_kl_kg_abl"]   for r in on_rows])
kg_sib = np.array([r["metadata_kl_kg_abl"]   for r in sib_rows])
de_sib = np.array([r["metadata_kl_dense_abl"] for r in sib_rows])

on_ci       = bootstrap_mean_ci(kg_on)
kg_col_ci   = bootstrap_mean_ci(kg_sib)
de_col_ci   = bootstrap_mean_ci(de_sib)
col_diff_ci = paired_bootstrap_diff(de_sib, kg_sib)   # dense - kg collateral (>0 => KG surgical)

print(f"Georgia / absorber 16009  (full edit, n_on={len(kg_on)} n_sib={len(kg_sib)}, B_BOOT={B_BOOT})")
print(f"  KG-ABL on-target KL   mean={on_ci['mean']:.5f}  CI[{on_ci['ci_lo']:.5f}, {on_ci['ci_hi']:.5f}]  excl_0={on_ci['excl_0']}")
print(f"  KG-ABL collateral KL  mean={kg_col_ci['mean']:.6f}  CI[{kg_col_ci['ci_lo']:.6f}, {kg_col_ci['ci_hi']:.6f}]")
print(f"  DENSE  collateral KL  mean={de_col_ci['mean']:.5f}  CI[{de_col_ci['ci_lo']:.5f}, {de_col_ci['ci_hi']:.5f}]")
print(f"  dense - kg collateral diff={col_diff_ci['diff']:.5f}  CI[{col_diff_ci['ci_lo']:.5f}, {col_diff_ci['ci_hi']:.5f}]  excl_0={col_diff_ci['excl_0']}")
print(f"  per-context selectivity (on / kg-collateral) = {on_ci['mean']/(kg_col_ci['mean']+EPS):.0f}x")

## Step 4 — Rebuild the regime **router map**

The firing-Jaccard / parent-recall-hole router splits cases into two regimes:
**absorption** (low firing-Jaccard, clean parent hole → *clean* surgical edits) vs. **co-firing**
(high firing-Jaccard, no hole → *not* cleanly surgical). Rebuilt from the per-case stats exactly
as `method.py`'s summary does.

In [ ]:
def _mean(xs):
    xs = [x for x in xs if x is not None]
    return float(np.mean(xs)) if xs else None

absn = [c for c in per_case if c.get("regime") == "absorption" and "matched" in c]
cofn = [c for c in per_case if c.get("regime") == "co-firing" and "matched" in c]
router = {}
for name, grp in (("absorption", absn), ("co_firing", cofn)):
    router[name] = {
        "n": len(grp),
        "mean_selectivity_ratio": _mean([c["headline_selectivity_ratio"] for c in grp]),
        "mean_firing_jaccard": _mean([c["firing_jaccard_with_parent"] for c in grp]),
        "mean_kg_offtarget_footprint": _mean([c["matched"]["KG-ABL"].get("token_footprint_offtarget") for c in grp]),
        "mean_kg_collateral": _mean([c["selectivity_CIs"]["KG-ABL_collateral"]["mean"] for c in grp]),
    }

for name, v in router.items():
    print(f"{name:<11} n={v['n']}  mean_selectivity_ratio={v['mean_selectivity_ratio']:>9.1f}  "
          f"firing_jaccard={v['mean_firing_jaccard']:.3f}  kg_footprint={v['mean_kg_offtarget_footprint']:.4f}")
split = router["absorption"]["mean_selectivity_ratio"] / max(router["co_firing"]["mean_selectivity_ratio"], EPS)
print(f"\n~{split:.0f}x selectivity split between the absorption and co-firing regimes")
print("\nHonest negatives reported by the method:")
for h in data["honest_negatives"]:
    print("  -", h[:160])

## Results & visualization

Three panels: (1) per-case surgical selectivity ratio (log scale; green = absorption regime, red =
co-firing), (2) the Georgia on-target vs collateral KL curves (the surgical signature — on-target
rises, collateral stays flat near 0), and (3) the regime router map (selectivity ratio vs
firing-Jaccard).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))

# (1) selectivity ratio per case (log scale)
labels = [f"{c['family'][:4]}/{c['target_subcontext']}\n#{c['absorber_latent']}" for c in per_case]
ratios = [c["headline_selectivity_ratio"] for c in per_case]
colors = ["#2ca02c" if c["regime"] == "absorption" else "#d62728" for c in per_case]
ax = axes[0]
ax.bar(range(len(per_case)), ratios, color=colors)
ax.set_yscale("log")
ax.axhline(RATIO_CLEAN_THRESHOLD, ls="--", color="gray", lw=1, label=f"clean threshold ({RATIO_CLEAN_THRESHOLD}x)")
ax.set_xticks(range(len(per_case))); ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
ax.set_ylabel("KG / DENSE selectivity ratio (log)")
ax.set_title("Surgical selectivity per case\n(green = absorption, red = co-firing)")
ax.legend(fontsize=8)

# (2) Georgia on-target vs collateral KL curves (drop scale=0 -> KL=0 for the log axis)
ax = axes[1]
g = per_case[0]; cur = g["curves"]; sc = cur["KG-ABL"]["scales"][1:]
ax.plot(sc, cur["KG-ABL"]["beh_on_target"][1:],  "o-",  color="#2ca02c", label="KG-ABL on-target")
ax.plot(sc, cur["KG-ABL"]["beh_collateral"][1:], "o--", color="#2ca02c", alpha=0.5, label="KG-ABL collateral")
ax.plot(sc, cur["DENSE-ABL"]["beh_on_target"][1:],  "s-",  color="#d62728", label="DENSE on-target")
ax.plot(sc, cur["DENSE-ABL"]["beh_collateral"][1:], "s--", color="#d62728", alpha=0.5, label="DENSE collateral")
ax.set_yscale("log"); ax.set_xlabel("edit scale (lambda / beta)"); ax.set_ylabel("behavioral next-token KL")
ax.set_title(f"{g['family']}/{g['target_subcontext']} (absorber {g['absorber_latent']})\non-target vs collateral KL")
ax.legend(fontsize=7)

# (3) regime router map: selectivity ratio vs firing-Jaccard
ax = axes[2]
for c in per_case:
    col = "#2ca02c" if c["regime"] == "absorption" else "#d62728"
    ax.scatter(c["firing_jaccard_with_parent"], c["headline_selectivity_ratio"], color=col, s=90, edgecolor="k", lw=0.5)
ax.set_yscale("log"); ax.set_xlabel("firing-Jaccard with parent"); ax.set_ylabel("selectivity ratio (log)")
ax.set_title("Regime router map\nlow Jaccard -> surgical | high Jaccard -> co-firing")

plt.tight_layout()
plt.savefig("m1b_demo_summary.png", dpi=110, bbox_inches="tight")
plt.show()
print("saved m1b_demo_summary.png")

# headline recap
s = data["summary"]
print(f"\nHEADLINE: {s['n_surgical_confirmed']}/{s['n_cases']} SURGICAL_EDIT_CONFIRMED | "
      f"Georgia selectivity ratio = {s['headline_selectivity_ratio']:.0f}x")